In [1]:
import pandas as pd
import math
from Bio import SeqIO
from modules.sequences import (
    split_train_val_test,
    encode_sequences,
    remove_unknown_amino_acids
)
from pathlib import Path
import joblib
from modules.expected_value import (
    generate_random_sequences,
    forward_evalue,
    calibrate_forward
)

In [2]:
family_df = pd.read_csv('protein_families.csv')
ave_len_recs = []

for i in range(len(family_df)):
    name, accession, match_len, seed_len, type_, clan = family_df.iloc[i]
    
    # read the fasta files
    recs = [(r.id, str(r.seq)) for r in SeqIO.parse(f"sequences/{accession}.fasta", "fasta")]

    train_seqs, val_seqs, test_seqs = split_train_val_test(recs)

    remove_train, train_seqs = remove_unknown_amino_acids(train_seqs)
    remove_val, val_seqs = remove_unknown_amino_acids(val_seqs)
    remove_test, test_seqs = remove_unknown_amino_acids(test_seqs)

    recs = train_seqs + val_seqs + test_seqs
    list_len_seq = [len(seq) for seq in recs]
    ave_len = sum(list_len_seq) / len(list_len_seq)
    std = math.sqrt(sum([(len_-ave_len)**2 for len_ in list_len_seq]) / len(list_len_seq))

    ave_len_recs.append([name, accession, ave_len, std])

ave_std_len = pd.DataFrame(ave_len_recs)
ave_std_len.columns = ['name', 'accession', 'ave_len', 'std_len']
ave_std_len

,name,accession,ave_len,std_len
0,DUF7563,PF24444,69.000000,24.111408
1,JmjN,PF02375,1031.940000,516.128372
2,KOW,PF00467,243.330000,244.983553
3,MerR,PF00376,189.252525,98.779046
4,SapB_2,PF03489,469.220000,345.405402
5,DnaJ_C_2,PF27439,378.950000,6.023911
6,DUF1232,PF06803,148.700000,57.516519
7,DUF1328,PF07043,57.130000,7.036555
8,DUF5808,PF19124,297.710000,129.128796
9,E3_binding,PF02817,479.800000,144.261221


In [3]:
import joblib
import numpy as np
import pandas as pd
import pyhmmer

from pathlib import Path
from Bio import SeqIO

symbols = [
    '^', 'A', 'C', 'D', 'E', 'F', 'G',
    'H', 'I', 'K', 'L', 'M', 'N',
    'P', 'Q', 'R', 'S', 'T', 'V',
    'W', 'Y', '$'
]

sym_idx = {
    symbol: j
    for j, symbol in enumerate(symbols)
}

BEGIN = sym_idx["^"]
END = sym_idx["$"]

# Initialize HMMER background model
alphabet = pyhmmer.easel.Alphabet.amino()
background = pyhmmer.plan7.Background(alphabet)

# Use HMMER's canonical amino-acid order.
amino_acids = alphabet.symbols[:alphabet.K]

freqs = np.asarray(
    background.residue_frequencies,
    dtype=float
)[:alphabet.K]

freqs = freqs / freqs.sum()

null_probs = dict(zip(amino_acids, freqs))


def null_log_likelihood(seq):
    return sum(
        np.log(null_probs[aa])
        for aa in seq
        if aa not in "^$"
    )


def forward_score(model, seq):
    X, _ = encode_sequences(sym_idx, [seq])

    log_like_m = model.score(X)
    log_like_r = null_log_likelihood(seq)

    return (log_like_m - log_like_r) / np.log(2)


model_path = Path("saved_models/phmm/main")

# =========================================
# STEP 1: Collect all test sequences
# =========================================

test_database = []
model_info = []

for i in range(len(family_df)):
    name, accession, match_len, seed_len, type_, clan = family_df.iloc[i]

    recs = [
        (r.id, str(r.seq))
        for r in SeqIO.parse(
            f"sequences/{accession}.fasta",
            "fasta"
        )
    ]

    _, _, test_seqs = split_train_val_test(recs)

    removed_count, test_seqs = remove_unknown_amino_acids(
        test_seqs
    )

    print(
        f"{name}_{accession}: "
        f"{len(test_seqs)} test sequences, "
        f"{removed_count} removed"
    )

    model_id = f"{name}_{accession}_m{match_len}"

    model_info.append({
        "model_id": model_id,
        "family": name,
        "accession": accession,
        "model_path": model_path / f"{model_id}.joblib"
    })

    for j, seq in enumerate(test_seqs):
        test_database.append({
            "sequence_id": f"{i}_{j}",
            "source_family": name,
            "source_accession": accession,
            "sequence": seq,
            "length": len(seq)
        })

# Total number of target sequences
D = len(test_database)

print(f"\nTotal test sequences: {D}")
print(f"Total models: {len(model_info)}")
print(f"Total comparisons: {D * len(model_info)}")

if D == 0:
    raise ValueError("The test database is empty.")

DUF7563_PF24444: 20 test sequences, 0 removed
JmjN_PF02375: 20 test sequences, 0 removed
KOW_PF00467: 20 test sequences, 0 removed
MerR_PF00376: 20 test sequences, 0 removed
SapB_2_PF03489: 20 test sequences, 0 removed
DnaJ_C_2_PF27439: 20 test sequences, 0 removed
DUF1232_PF06803: 20 test sequences, 0 removed
DUF1328_PF07043: 20 test sequences, 0 removed
DUF5808_PF19124: 20 test sequences, 0 removed
E3_binding_PF02817: 20 test sequences, 0 removed
FATC_PF02260: 19 test sequences, 1 removed
Histone_H2A_C_PF16211: 20 test sequences, 0 removed
LeuD_C_PF27434: 20 test sequences, 0 removed
MutS_C_PF27441: 20 test sequences, 0 removed
Nuo9p5_PF30687: 20 test sequences, 0 removed
Ribosomal_uL15_N_PF29471: 20 test sequences, 0 removed
Se-cys_synth_N_PF12390: 20 test sequences, 0 removed
Sigma70_r1_2_PF00140: 20 test sequences, 0 removed
Somatomedin_B_PF01033: 20 test sequences, 0 removed
YqhI_PF23678: 20 test sequences, 0 removed

Total test sequences: 399
Total models: 20
Total comparisons: 

In [5]:
# =========================================
# STEP 2: Evaluate each model against
#         the entire test database
# =========================================

results = []

for info in model_info:
    model_id = info["model_id"]

    print(f"\nEvaluating model: {model_id}")

    model = joblib.load(info["model_path"])

    # Each model requires its own calibrated thresholds.
    threshold_cache = {}

    for entry in test_database:
        seq = entry["sequence"]
        length = entry["length"]

        # Estimate threshold for each unique length
        if length not in threshold_cache:
            random_seqs = generate_random_sequences(
                null_probs,
                n=200,
                length=length,
                seed=42
            )

            random_scores = [
                forward_score(model, random_seq)
                for random_seq in random_seqs
            ]

            if not np.all(np.isfinite(random_scores)):
                raise ValueError(
                    f"Non-finite calibration scores for "
                    f"{model_id}, length={length}"
                )

            threshold_cache[length] = calibrate_forward(
                random_scores
            )

        t = threshold_cache[length]

        # Calculate Forward score
        score = forward_score(model, seq)

        # Calculate approximate E-value
        evalue = forward_evalue(
            score=score,
            t=t,
            database_size=D
        )

        results.append({
            "model_id": model_id,
            "model_family": info["family"],
            "sequence_id": entry["sequence_id"],
            "source_family": entry["source_family"],
            "source_accession": entry["source_accession"],
            "length": length,
            "same_family": (
                info["accession"]
                == entry["source_accession"]
            ),
            "score": score,
            "t": t,
            "evalue": evalue
        })


# =========================================
# STEP 3: Save results
# =========================================

results_df = pd.DataFrame(results)
database_df = pd.DataFrame(test_database)

results_df.to_csv(
    "all_family_evalues.csv",
    index=False
)

database_df.to_csv(
    "test_database.csv",
    index=False
)

# E-value matrix: sequences x models
evalue_matrix = results_df.pivot(
    index="sequence_id",
    columns="model_id",
    values="evalue"
)

evalue_matrix.to_csv("evalue_matrix.csv")

# Forward-score matrix
score_matrix = results_df.pivot(
    index="sequence_id",
    columns="model_id",
    values="score"
)

score_matrix.to_csv("score_matrix.csv")

print("\nEvaluation completed!")
print(results_df.head())

In [19]:
import torch

# # Check for NVIDIA GPU (CUDA)
# print("CUDA Available:", torch.cuda.is_available())

# # Check for Apple Silicon Mac GPU (MPS)
# print("MPS Available:", torch.backends.mps.is_available())

# if torch.cuda.is_available():
#     device = torch.device("cuda")
# elif torch.backends.mps.is_available():
#     device = torch.device("mps")
# else:
#     device = torch.device("cpu")

# print(f"Using device: {device}")
# device = 'cpu'

In [21]:

import time
import torch
import numpy as np
import pandas as pd
import pyhmmer

from pathlib import Path
from scipy.stats import gumbel_r
from modules.model import init_emission_mat


# =========================================
# STEP 1: Configuration
# =========================================

family_df = pd.read_csv("protein_families.csv")

model_path = Path("saved_models/phqmm/main")
output_dir = Path("results/phqmm/evalues")
output_dir.mkdir(parents=True, exist_ok=True)

device = "cpu"

# Same test database used for PHMM evaluation
test_database = pd.read_csv(
    "test_database.csv"
).to_dict("records")

D = len(test_database)

if D == 0:
    raise ValueError("Test database is empty.")

# Calibration settings
N_NULL = 200
SEED = 42

# Verbose progress settings
VERBOSE = True
RANDOM_PRINT_EVERY = 10
TEST_PRINT_EVERY = 1


# =========================================
# STEP 2: Initialize HMMER background model
# =========================================

amino_acids = "ACDEFGHIKLMNPQRSTVWY"

alphabet = pyhmmer.easel.Alphabet.amino()
background = pyhmmer.plan7.Background(alphabet)

null_probs = dict(
    zip(
        amino_acids,
        background.residue_frequencies
    )
)


# =========================================
# STEP 3: PHQMM Forward log-likelihood
# =========================================

@torch.no_grad()
def hqmm_log_likelihood(p, K, W, sym_idx, seq):
    """
    Compute the log-likelihood of a sequence
    using the HQMM Forward recursion.

    Excludes ^ and includes $ exactly once.
    """

    seq = str(seq).removeprefix("^").removesuffix("$")

    encoded = [
        sym_idx[aa]
        for aa in seq
    ] + [sym_idx["$"]]

    rho = p.clone()

    loglik = 0.0

    for symbol in encoded:

        rho_next = torch.zeros_like(rho)

        for w in range(W):
            Kw = K[(w, symbol)]

            rho_next += (
                Kw @ rho @ Kw.conj().T
            )

        prob = torch.trace(rho_next).real.item()

        if not np.isfinite(prob) or prob <= 0:
            return -np.inf

        loglik += np.log(prob)

        # Normalize conditional density matrix
        rho = rho_next / prob

    return float(loglik)


def hqmm_forward_score(p, K, W, sym_idx, seq):
    """
    Compute the PHQMM Forward log-odds score
    in bits.
    """

    model_loglik = hqmm_log_likelihood(
        p, K, W, sym_idx, seq
    )

    null_loglik = null_log_likelihood(seq)

    return (
        model_loglik - null_loglik
    ) / np.log(2.0)


# =========================================
# STEP 5: Collect saved PHQMM models
# =========================================

model_info = []

for i in range(len(family_df)):

    name, accession, match_len, seed_len, type_, clan = (
        family_df.iloc[i]
    )

    n = match_len * 2 + 1

    model_name = (
        f"{name}_{accession}_"
        f"m{match_len}_n{n}_hqmm.pt"
    )

    checkpoint_path = model_path / model_name

    if not checkpoint_path.exists():
        print(
            f"Missing checkpoint: {checkpoint_path}",
            flush=True
        )
        continue

    model_info.append({
        "model_id": f"{name}_{accession}",
        "family": name,
        "accession": accession,
        "match_len": match_len,
        "n": n,
        "model_path": checkpoint_path
    })

print(f"Models found: {len(model_info)}", flush=True)
print(f"Test sequences: {D}", flush=True)
print(f"Device: {device}", flush=True)


# =========================================
# STEP 6: Cache random sequences by length
# =========================================

# Generate exactly the same background sequences
# for a given length across all PHQMM models.

random_sequence_cache = {}

unique_lengths = sorted({
    int(entry["length"])
    for entry in test_database
})

for length in unique_lengths:

    random_sequence_cache[length] = (
        generate_random_sequences(
            null_probs,
            n=N_NULL,
            length=length,
            seed=SEED
        )
    )

print(
    f"Unique sequence lengths: "
    f"{len(unique_lengths)}",
    flush=True
)

print(
    f"Calibration sequences per model: "
    f"{len(unique_lengths) * N_NULL}",
    flush=True
)


# =========================================
# STEP 7: Evaluate each PHQMM against
#         the entire test database
# =========================================

results = []

overall_start = time.perf_counter()

for model_idx, info in enumerate(model_info, start=1):

    model_start = time.perf_counter()

    model_id = info["model_id"]

    print("\n" + "=" * 65, flush=True)
    print(
        f"[MODEL {model_idx}/{len(model_info)}] "
        f"Evaluating model: {model_id}",
        flush=True
    )
    print("=" * 65, flush=True)

    # -------------------------------------
    # Load saved PHQMM
    # -------------------------------------

    print("Loading PHQMM checkpoint...", flush=True)

    checkpoint = torch.load(
        info["model_path"],
        map_location=device,
        weights_only=False
    )

    K = checkpoint["K"]
    p = checkpoint["p"]
    W = int(checkpoint["W"])

    K = {
        key: value.to(device=device)
        for key, value in K.items()
    }

    K_ref = next(iter(K.values()))

    p = p.to(
        device=K_ref.device,
        dtype=K_ref.dtype
    )

    print(
        f"Checkpoint loaded | "
        f"n={info['n']} | "
        f"W={W} | "
        f"Kraus operators={len(K)}",
        flush=True
    )

    # -------------------------------------
    # Recover symbol indexing
    # -------------------------------------

    emit_matdf = init_emission_mat(
        info["match_len"],
        seed=2
    )

    symbols = list(emit_matdf.index)

    sym_idx = {
        symbol: j
        for j, symbol in enumerate(symbols)
    }

    # -------------------------------------
    # Each model requires its own
    # calibrated thresholds
    # -------------------------------------

    threshold_cache = {}

    for entry_idx, entry in enumerate(
        test_database, start=1
    ):

        seq = entry["sequence"]
        length = int(entry["length"])

        # ---------------------------------
        # Calibrate each unique length
        # ---------------------------------

        if length not in threshold_cache:

            length_start = time.perf_counter()

            random_seqs = (
                random_sequence_cache[length]
            )

            length_idx = len(threshold_cache) + 1

            print(
                f"\n[CALIBRATION {length_idx}/"
                f"{len(unique_lengths)}] "
                f"Model={model_id} | "
                f"Length={length}",
                flush=True
            )

            random_scores = []

            for random_idx, random_seq in enumerate(
                random_seqs, start=1
            ):

                # Print the current random sequence
                # BEFORE evaluating it.
                if VERBOSE and (
                    random_idx == 1
                    or random_idx % RANDOM_PRINT_EVERY == 0
                    or random_idx == len(random_seqs)
                ):
                    print(
                        f"  [RANDOM "
                        f"{random_idx}/{len(random_seqs)}] "
                        f"L={length} | "
                        f"Seq={random_seq[:30]}...",
                        flush=True
                    )

                random_score = hqmm_forward_score(
                    p,
                    K,
                    W,
                    sym_idx,
                    random_seq
                )

                random_scores.append(random_score)

            if not np.all(
                np.isfinite(random_scores)
            ):
                raise ValueError(
                    f"Non-finite calibration scores "
                    f"for {model_id}, length={length}"
                )

            threshold_cache[length] = (
                calibrate_forward(random_scores)
            )

            length_elapsed = (
                time.perf_counter() - length_start
            )

            print(
                f"[CALIBRATION COMPLETE] "
                f"Length={length} | "
                f"t={threshold_cache[length]:.4f} | "
                f"Elapsed={length_elapsed:.2f}s",
                flush=True
            )

        t = threshold_cache[length]

        # ---------------------------------
        # Print current test sequence
        # ---------------------------------

        if VERBOSE and (
            entry_idx == 1
            or entry_idx % TEST_PRINT_EVERY == 0
            or entry_idx == D
        ):
            print(
                f"\n[TEST {entry_idx}/{D}] "
                f"Model={model_id} | "
                f"ID={entry['sequence_id']} | "
                f"Length={length} | "
                f"Seq={str(seq)[:30]}...",
                flush=True
            )

        # ---------------------------------
        # Calculate PHQMM Forward score
        # ---------------------------------

        score = hqmm_forward_score(
            p,
            K,
            W,
            sym_idx,
            seq
        )

        # ---------------------------------
        # Calculate approximate E-value
        # ---------------------------------

        evalue = forward_evalue(
            score=score,
            t=t,
            database_size=D
        )

        # Print score and E-value
        if VERBOSE and (
            entry_idx == 1
            or entry_idx % TEST_PRINT_EVERY == 0
            or entry_idx == D
        ):
            print(
                f"  Score={score:.4f} bits | "
                f"E-value={evalue:.4e}",
                flush=True
            )

        # ---------------------------------
        # Record result
        # ---------------------------------

        results.append({
            "model_id": model_id,
            "model_family": info["family"],
            "sequence_id": entry["sequence_id"],
            "source_family": entry["source_family"],
            "source_accession": entry["source_accession"],
            "length": length,
            "same_family": (
                info["accession"]
                == entry["source_accession"]
            ),
            "score": score,
            "t": t,
            "evalue": evalue
        })

    model_elapsed = (
        time.perf_counter() - model_start
    )

    print("\n" + "-" * 65, flush=True)
    print(
        f"[MODEL COMPLETE] {model_id}",
        flush=True
    )
    print(
        f"Test sequences evaluated: {D}",
        flush=True
    )
    print(
        f"Lengths calibrated: "
        f"{len(threshold_cache)}",
        flush=True
    )
    print(
        f"Model elapsed time: "
        f"{model_elapsed:.2f}s",
        flush=True
    )
    print(
        f"Models completed: "
        f"{model_idx}/{len(model_info)}",
        flush=True
    )
    print("-" * 65, flush=True)

    # Save intermediate results after each model
    pd.DataFrame(results).to_csv(
        output_dir / "all_family_evalues.csv",
        index=False
    )

    print(
        f"Intermediate results saved: "
        f"{len(results)} comparisons",
        flush=True
    )


# =========================================
# STEP 8: Save final CSV results
# =========================================

if not results:
    raise ValueError("No PHQMM models were evaluated.")

results_df = pd.DataFrame(results)

database_df = pd.DataFrame(test_database)

# All model-sequence scores and E-values
results_df.to_csv(
    output_dir / "all_family_evalues.csv",
    index=False
)

# Test database
database_df.to_csv(
    output_dir / "test_database.csv",
    index=False
)

# Model information
pd.DataFrame(model_info).assign(
    model_path=lambda df: df["model_path"].astype(str)
).to_csv(
    output_dir / "model_info.csv",
    index=False
)

# E-value matrix: sequences x models
evalue_matrix = results_df.pivot(
    index="sequence_id",
    columns="model_id",
    values="evalue"
)

evalue_matrix.to_csv(
    output_dir / "evalue_matrix.csv"
)

# Forward-score matrix
score_matrix = results_df.pivot(
    index="sequence_id",
    columns="model_id",
    values="score"
)

score_matrix.to_csv(
    output_dir / "score_matrix.csv"
)

overall_elapsed = (
    time.perf_counter() - overall_start
)

print("\n" + "=" * 65, flush=True)
print("PHQMM EVALUATION COMPLETED!", flush=True)
print(f"Models evaluated: {len(model_info)}", flush=True)
print(f"Test sequences: {D}", flush=True)
print(f"Total comparisons: {len(results)}", flush=True)
print(f"Total elapsed: {overall_elapsed:.2f}s", flush=True)
print(f"Output directory: {output_dir}", flush=True)
print("=" * 65, flush=True)

print(results_df.head())


Models found: 20
Test sequences: 399
Device: cpu
Unique sequence lengths: 274
Calibration sequences per model: 54800

[MODEL 1/20] Evaluating model: DUF7563_PF24444
Loading PHQMM checkpoint...
Checkpoint loaded | n=79 | W=2 | Kraus operators=42

[CALIBRATION 1/274] Model=DUF7563_PF24444 | Length=84
  [RANDOM 1/200] L=84 | Seq=SKTQDYRSDKIVPSKFLASPRIYVSELAEQ...
  [RANDOM 10/200] L=84 | Seq=HVAKQEADVAEADNFMNTATNESTGVKNYI...
  [RANDOM 20/200] L=84 | Seq=EESGVDQSFGWGKKFSSISEAMYAELSCIR...
  [RANDOM 30/200] L=84 | Seq=NPKRYEELLMAIMNPRPSNLNDGLTQCNFR...
  [RANDOM 40/200] L=84 | Seq=LLHDPAGDEPPKCKIRMLLWRLLAVGGYRA...
  [RANDOM 50/200] L=84 | Seq=NTGLVSTIYELLSVRPKLLALTQSILAILG...
  [RANDOM 60/200] L=84 | Seq=TFIREELFFAIGANPAGTMKLESVVFGEGH...
  [RANDOM 70/200] L=84 | Seq=LVPIKNRMLRQSARLAQRLSGEIKQTREIE...
  [RANDOM 80/200] L=84 | Seq=DQSIASEHPDAAVGPYPLLINWSKHTDTGR...
  [RANDOM 90/200] L=84 | Seq=LGKVLVLSGSGAEAVYRDVRHLPALREKLR...
  [RANDOM 100/200] L=84 | Seq=VTGKALAVTEQMSLEPCMEQSDIREMPISL...
  [RAND

KeyboardInterrupt: 